# Trabalho Final — Avaliação de Políticas Públicas com Dados (APPD)

**Programa:** Mestrado Profissional em Administração Pública — IDP
**Professor:** Hélio Bomfim de Macêdo Filho
**Integrantes:** [PREENCHER: nomes completos]
**Política avaliada:** Programa Academia da Saúde (Portaria GM/MS nº 719/2011; Portaria GM/MS nº 2.681/2013)

**Pergunta causal:** a implantação do primeiro polo do Programa Academia da Saúde reduziu as internações por doenças cardiovasculares e metabólicas sensíveis à atividade física (doenças hipertensivas, AVC e diabetes) nos municípios que o adotaram?

**Estimando:** ATT — efeito médio sobre os municípios que implantaram ao menos um polo.
**Unidade de análise:** município × ano, 2008–2024.
**Métodos:** escore de propensão (Módulo 03) e diferenças-em-diferenças com estudo de eventos, TWFE e Callaway & Sant'Anna (Módulo 04).

> **Declaração de uso de IA (versão preliminar, a atualizar na Fase 8).** Este caderno foi desenvolvido com apoio do Claude Code (Anthropic), modelo [PREENCHER: nome e versão], para gerar e depurar código. Os autores revisaram cada célula, reexecutaram integralmente o caderno e assumem responsabilidade pelo conteúdo. Prompts e saídas relevantes estão em `docs/ia/`.

**Caderno 01 — extração:** download, seleção, dicionários, agregação, validação e gravação em `data/processed/`.

**Localização do repositório.** Garante que o caderno rode a partir de `notebooks/`, inclusive quando aberto isolado (ex.: Google Colab), caso em que o repositório público é clonado. Todos os caminhos seguintes são relativos à raiz.

In [40]:
# Localiza a raiz do repositório (pasta com requirements.txt e src/appd).
# Se o caderno foi aberto isolado (ex.: Google Colab), clona o repositório
# público na pasta atual. Depois disso, tudo usa caminhos relativos à raiz.
import os
import subprocess
import sys
from pathlib import Path

URL_REPOSITORIO = "https://github.com/bsousa7/APPD_26.git"


def localizar_raiz():
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / "requirements.txt").exists() and (pasta / "src" / "appd").exists():
            return pasta
    return None


raiz = localizar_raiz()
if raiz is not None and "google.colab" in sys.modules and (raiz / ".git").exists():
    # No Colab, a cópia clonada pode estar desatualizada: atualiza sempre.
    subprocess.run(["git", "-C", str(raiz), "pull", "--ff-only", "-q"], check=True)
if raiz is None:
    copia = Path("APPD_26")
    if (copia / ".git").exists():
        subprocess.run(["git", "-C", str(copia), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(copia)], check=True)
    os.chdir(copia)
    if localizar_raiz() is None:
        raise RuntimeError(
            f"A cópia em {Path.cwd()} não tem requirements.txt e src/appd. "
            "Apague a pasta APPD_26 e execute esta célula de novo."
        )
os.chdir(localizar_raiz() / "notebooks")
versao = subprocess.run(["git", "log", "-1", "--format=%h %cs"], capture_output=True, text=True).stdout.strip()
print(f"Pasta de trabalho: {Path.cwd().parent.name}/notebooks | versão do repositório: {versao}")

# O Colab executa as células da aba aberta, não as da cópia do repositório.
# Se o caderno do repositório mudou, esta aba está desatualizada.
import json
import re

NOME_CADERNO = "01_extracao.ipynb"
VERSAO_CADERNO = "e8ad903e4d"
copia = Path.cwd() / NOME_CADERNO
if copia.exists():
    celulas = json.loads(copia.read_text(encoding="utf-8"))["cells"]
    fonte = "".join("".join(c["source"]) if isinstance(c["source"], list) else c["source"] for c in celulas)
    no_repositorio = re.search(r'VERSAO_CADERNO = "([0-9a-f]+)"', fonte)
    if no_repositorio and no_repositorio.group(1) != VERSAO_CADERNO:
        raise RuntimeError(
            f"Este caderno (versão {VERSAO_CADERNO}) está desatualizado: a versão no repositório é "
            f"{no_repositorio.group(1)}. Feche esta aba e abra o caderno de novo pelo link do README. "
            "O que já foi extraído para o Drive é reaproveitado.")
print("Versão do caderno:", VERSAO_CADERNO)

Pasta de trabalho: APPD_26/notebooks | versão do repositório: ec5da35 2026-11-24
Versão do caderno: e8ad903e4d


A saída mostra a pasta de trabalho e a versão do repositório em uso.

## 1. Identificação

Este caderno cobre as etapas 3 a 6 das orientações (download, descompressão e seleção, dicionários, construção e validação) e grava as bases agregadas usadas pelo caderno `02_analise.ipynb`.

| Base | Uso | Recorte |
|---|---|---|
| IBGE, API de localidades | lista de municípios, UF e região | atual |
| CNES-ST (estabelecimentos) | tratamento: polos tipo 74 | dezembro de cada ano |
| CNES-IN (incentivos) | tratamento alternativo R1 (81.12/81.13) | dezembro de cada ano |
| CNES-EP (equipes) e CNES-LT (leitos) | covariáveis | dezembro de 2010 |
| SIH/SUS-RD (AIH reduzida) | desfechos e placebo | todos os meses |
| IBGE/SIDRA 6579, 5938 e 1378 | população, PIB 2010, população idosa 2010 | anual / 2010 |

**Como executar.** Primeiro com `MODO_TESTE = True` (Acre, 2014–2015, poucos minutos): confira os números indicados na seção 6 com o TabNet. Depois mude para `MODO_TESTE = False` e execute tudo de novo. A extração completa leva várias horas, sobretudo o SIH; cada parte é gravada assim que termina, e uma nova execução retoma de onde parou. No Colab, os resultados ficam no Google Drive, na pasta `APPD_26_dados`.

**Tempo estimado da extração completa:** [PREENCHER após a primeira execução completa].

## 2. Preparação do ambiente

**O que:** instala as dependências com as versões fixadas em `requirements.txt`, importa as bibliotecas, define caminhos relativos à raiz do repositório e a semente aleatória.
**Por quê:** reprodutibilidade — qualquer pessoa deve conseguir rodar o caderno com "Restart & Run All" a partir de `notebooks/`.

No Colab, a célula também pede autorização para montar o Google Drive.

In [41]:
# Escolhe o arquivo de requisitos conforme o ambiente:
# - Google Colab: requirements-colab.txt (só o que o Colab não traz, sem
#   substituir pandas, ipykernel etc., que o próprio Colab exige);
# - ambiente local: requirements.txt (todas as bibliotecas, versões fixadas).
# O caminho é calculado a partir de notebooks/ e passado já resolvido ao %pip.
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
nome = "requirements-colab.txt" if NO_COLAB else "requirements.txt"
ARQ_REQUISITOS = (Path.cwd().parent / nome).resolve()
if not ARQ_REQUISITOS.exists():
    raise FileNotFoundError(
        f"{nome} não encontrado a partir de {Path.cwd().name}/. "
        "Execute antes a célula 'Localização do repositório' (seção 1)."
    )
print("Instalando", nome)
%pip install -q -r {ARQ_REQUISITOS}

# Se o pip trocou a versão de uma biblioteca que já estava carregada (comum no
# Colab, que pré-carrega numpy e pandas), a sessão precisa ser reiniciada.
from importlib.metadata import version as versao_instalada

trocadas = [
    f"{nome} {sys.modules[nome].__version__} -> {versao_instalada(nome)}"
    for nome in ["numpy", "pandas", "scipy"]
    if nome in sys.modules and sys.modules[nome].__version__ != versao_instalada(nome)
]
if trocadas:
    raise RuntimeError(
        "Bibliotecas atualizadas: " + "; ".join(trocadas) + ". Reinicie a sessão "
        "(Ambiente de execução > Reiniciar sessão) e execute TUDO de novo, desde a "
        "primeira célula de código. Na segunda execução esta célula passa direto."
    )
print("Dependências prontas.")

Instalando requirements-colab.txt
Dependências prontas.


Dependências instaladas. Se a célula pedir para reiniciar a sessão, reinicie e execute tudo desde a primeira célula de código.

In [42]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

RAIZ = Path("..")
if not (RAIZ / "src" / "appd").exists():
    raise RuntimeError(
        "Pasta de trabalho errada (comum após reiniciar a sessão do Colab). "
        "Execute o caderno desde a primeira célula de código (localização do repositório)."
    )
sys.path.insert(0, str(RAIZ / "src"))
# Descarta versões já carregadas do pacote do projeto: se a primeira célula
# atualizou o código (git pull), a importação abaixo usa a versão nova.
for _m in [m for m in sys.modules if m == "appd" or m.startswith("appd.")]:
    del sys.modules[_m]

DIR_RAW = RAIZ / "data" / "raw"
DIR_PROC = RAIZ / "data" / "processed"
DIR_DIC = DIR_PROC / "dicionarios"
DIR_FIG = RAIZ / "outputs" / "figuras"
DIR_TAB = RAIZ / "outputs" / "tabelas"
for d in (DIR_RAW, DIR_PROC, DIR_DIC, DIR_FIG, DIR_TAB):
    d.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 30)
from importlib.metadata import version
print("Python", sys.version.split()[0], "|", ", ".join(
    f"{p} {version(p)}" for p in ["pandas", "numpy", "scipy", "statsmodels",
                                  "scikit-learn", "linearmodels", "differences", "pysus"]))

import time

from appd import agregacao as ag
from appd import download as dl
from appd.armazenamento import com_checkpoint, extrair_partes, preparar_armazenamento

# No Colab, data/processed/ passa a apontar para uma pasta do Google Drive
# (os resultados sobrevivem ao fim da sessão). Fora do Colab, nada muda.
DIR_PROC = preparar_armazenamento(RAIZ)
dl.configurar_cache(RAIZ)

# ---------------------------------------------------------------- recorte
# MODO_TESTE = True roda só o Acre (AC) em 2014–2015, em poucos minutos, para
# conferir os números com o TabNet antes da extração nacional (várias horas).
MODO_TESTE = False

UFS_TODAS = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA",
             "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]
UFS = ["AC"] if MODO_TESTE else UFS_TODAS
ANOS = [2014, 2015] if MODO_TESTE else list(range(2008, 2025))
SIMULTANEOS = 3              # partes (UFs) baixadas ao mesmo tempo
MES_CNES = 12                # uma competência por ano (dezembro), PLANO.md seção 5
ANO_COVARIAVEIS = 2010       # covariáveis pré-tratamento
ANO_PRIMEIRA_COORTE = 2013   # decisão de 22/11/2026 (docs/decisoes.md)

DIR_SAIDA = DIR_PROC / "teste" if MODO_TESTE else DIR_PROC
DIR_PARCIAIS = DIR_SAIDA / "parciais"
DIR_DIC = DIR_PROC / "dicionarios"
for d in (DIR_SAIDA, DIR_PARCIAIS, DIR_DIC):
    d.mkdir(parents=True, exist_ok=True)

FALHAS = []  # partes que falharam; são refeitas na próxima execução
INICIO = time.time()
print("Modo:", "TESTE (AC, 2014–2015)" if MODO_TESTE else "COMPLETO", "| UFs:", len(UFS),
      "| anos:", ANOS[0], "a", ANOS[-1])

Python 3.13.18 | pandas 2.2.3, numpy 2.4.6, scipy 1.16.3, statsmodels 0.15.0, scikit-learn 1.6.1, linearmodels 7.1, differences 0.3.0, pysus 2.12.2
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dados processados no Google Drive: MyDrive/APPD_26_dados/processed
Modo: COMPLETO | UFs: 27 | anos: 2008 a 2024


O recorte em uso aparece na última linha. `FALHAS` acumula as partes que não puderam ser baixadas.

## 3. Download dos dados brutos

Cada base é baixada da fonte original por `src/appd/download.py` (FTP do DATASUS via PySUS ou APIs do IBGE), com a origem efetiva e a data de acesso registradas. Os arquivos do DATASUS são lidos só nas colunas necessárias, agregados na hora e apagados (seção 4).

A função abaixo organiza a retomada e o paralelismo: até `SIMULTANEOS` partes (por exemplo, três UFs em um mesmo ano) são baixadas ao mesmo tempo, cada uma com até 3 tentativas, e cada parte é gravada em `parciais/` assim que termina. Se o FTP do DATASUS recusar conexões simultâneas (muitas falhas na mesma base), reduza `SIMULTANEOS` para 1 e execute de novo.

In [43]:
def extrair(nome, chaves, arquivo, gerar):
    """Executa `gerar(*chave)` para cada chave, com checkpoint em `arquivo(*chave)`.

    Roda até SIMULTANEOS partes ao mesmo tempo (UFs diferentes), com até 3
    tentativas por parte. Partes já gravadas são lidas do disco; uma parte que
    falha em todas as tentativas vai para FALHAS e é refeita na próxima execução.
    A implementação está em src/appd/armazenamento.py (extrair_partes).
    """
    return extrair_partes(nome, chaves, arquivo, gerar, FALHAS, simultaneos=SIMULTANEOS)

A função está definida; ela é usada em todas as bases do DATASUS a seguir.

### 3.1 Lista de municípios (API de localidades do IBGE)
**Por quê:** chave mestra do painel (código de 7 dígitos, `cod6` do DATASUS, UF e região).

In [44]:
municipios = com_checkpoint(DIR_SAIDA / "municipios.parquet",
                            lambda: dl.municipios_ibge().query("uf in @UFS").reset_index(drop=True))
assert municipios["cod_ibge7"].is_unique and municipios["cod6"].is_unique
assert municipios[["cod_ibge7", "cod6", "uf", "regiao"]].notna().all().all()
print(len(municipios), "municípios em", municipios["uf"].nunique(), "UFs")
municipios.head(3)

5571 municípios em 27 UFs


,cod_ibge7,cod6,municipio,uf,regiao
0,1100015,110001,Alta Floresta D'Oeste,RO,Norte
1,1100023,110002,Ariquemes,RO,Norte
2,1100031,110003,Cabixi,RO,Norte


**Resultado:** [PREENCHER: número de municípios. A API devolve a malha atual (5.571 no país), que inclui municípios instalados depois de 2008; eles são tratados na seção 6.]

### 3.2 CNES-ST: polos tipo 74 por município × ano
**Por quê:** constrói o tratamento D. Para cada UF e ano, lê só `CNES`, `CODUFMUN` e `TP_UNID` (sem `CPF_CNPJ`) da competência de dezembro e conta os estabelecimentos tipo 74.

In [45]:
COLS_ST = ["CNES", "CODUFMUN", "TP_UNID"]


def polos_uf_ano(uf, ano):
    st = dl.baixar_datasus("cnes", uf, ano, MES_CNES, grupo="ST", colunas=COLS_ST)
    assert set(COLS_ST) <= set(st.columns), f"ST {uf} {ano}: colunas ausentes"
    return ag.contar_polos(st, ano)


polos = extrair("CNES-ST", [(uf, ano) for ano in ANOS for uf in UFS],
                lambda uf, ano: DIR_PARCIAIS / "cnes_st" / f"ST_{uf}_{ano}.parquet", polos_uf_ano)
if len(polos):
    assert not polos.duplicated(["cod6", "ano"]).any()
    display(polos.groupby("ano")["n_polos"].agg(municipios="count", polos="sum").T)

CNES-ST: 459 de 459 partes já gravadas (lidas do disco); 0 a baixar
CNES-ST: 25/459 partes | 0.0 min | falhas: 0
CNES-ST: 50/459 partes | 0.0 min | falhas: 0
CNES-ST: 75/459 partes | 0.0 min | falhas: 0
CNES-ST: 100/459 partes | 0.0 min | falhas: 0
CNES-ST: 125/459 partes | 0.0 min | falhas: 0
CNES-ST: 150/459 partes | 0.0 min | falhas: 0
CNES-ST: 175/459 partes | 0.0 min | falhas: 0
CNES-ST: 200/459 partes | 0.0 min | falhas: 0
CNES-ST: 225/459 partes | 0.0 min | falhas: 0
CNES-ST: 250/459 partes | 0.0 min | falhas: 0
CNES-ST: 275/459 partes | 0.0 min | falhas: 0
CNES-ST: 300/459 partes | 0.0 min | falhas: 0
CNES-ST: 325/459 partes | 0.0 min | falhas: 0
CNES-ST: 350/459 partes | 0.0 min | falhas: 0
CNES-ST: 375/459 partes | 0.0 min | falhas: 0
CNES-ST: 400/459 partes | 0.0 min | falhas: 0
CNES-ST: 425/459 partes | 0.0 min | falhas: 0
CNES-ST: 450/459 partes | 0.0 min | falhas: 0
CNES-ST: 459/459 partes | 0.0 min | falhas: 0


ano,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
municipios,300,308,508,896,1241,1503,1746,1963,2109,2238,2367,2455,2480,2508
polos,464,468,735,1182,1560,1907,2216,2538,2764,2974,3191,3362,3401,3491


**Resultado:** [PREENCHER: municípios e polos tipo 74 por ano; no modo completo, conferir com a varredura do `00_teste_acesso` (300 municípios em 2011, 508 em 2013, 1.503 em 2016)]

### 3.3 CNES-IN: incentivos de custeio (robustez R1)
**Por quê:** tratamento alternativo, "polo com incentivo de custeio" (códigos 81.12/81.13).

**Layout observado** (CNES-IN do AC, dez/2015, executado no Colab em 22/11/2026): o código da habilitação/incentivo está em `SGRUPHAB`, e a vigência em `CMPT_INI` e `CMPT_FIM`. Conta-se cada estabelecimento com incentivo **vigente** na competência de dezembro. A célula mostra os códigos mais frequentes (contagens agregadas) para conferir se 81.12 e 81.13 aparecem com esse formato. Lê só `CNES`, `CODUFMUN`, `SGRUPHAB`, `CMPT_INI` e `CMPT_FIM` (o arquivo também traz `CPF_CNPJ`, que não é lido).

**Decisão:** para algumas UF-anos (ex.: RR, dez/2008) o arquivo CNES-IN não existe no FTP do DATASUS nem no espelho do PySUS. Esses casos são tratados como **nenhum estabelecimento com incentivo** e listados em `cnes_in_arquivos_inexistentes.csv`. Falhas de outro tipo (rede, por exemplo) continuam sendo falhas e são refeitas.

In [46]:
# Diagnóstico de 22/11/2026 (seção 3.3.1): 8112 está em 95 estabelecimentos em dez/2016,
# 94 deles polos tipo 74; 8113 não aparece entre os vigentes. Confirmar na documentação.
CODIGOS_INCENTIVO = {"81.12", "81.13"}
COL_INCENTIVO = "SGRUPHAB"
COLS_IN = ["CNES", "CODUFMUN", COL_INCENTIVO, "CMPT_INI", "CMPT_FIM"]
frequencias_in = []


INEXISTENTES_IN = []


def incentivos_uf_ano(uf, ano):
    try:
        inc = dl.baixar_datasus("cnes", uf, ano, MES_CNES, grupo="IN", colunas=COLS_IN)
    except dl.ArquivoInexistente:
        # Sem arquivo de incentivos para a UF-ano: nenhum estabelecimento com incentivo.
        INEXISTENTES_IN.append({"uf": uf, "ano": ano})
        return pd.DataFrame({"cod6": pd.Series(dtype="string"), "ano": pd.Series(dtype="int64"),
                             "n_estab_incentivo": pd.Series(dtype="int64")})
    assert {"CNES", "CODUFMUN", COL_INCENTIVO} <= set(inc.columns), f"IN {uf} {ano}: {list(inc.columns)}"
    if uf == UFS[0] and ano == ANOS[-1]:
        frequencias_in.append(inc[COL_INCENTIVO].value_counts().head(15))
    return ag.contar_incentivos(inc, ano, CODIGOS_INCENTIVO, COL_INCENTIVO,
                                competencia=ano * 100 + MES_CNES)


incentivos = extrair("CNES-IN", [(uf, ano) for ano in ANOS for uf in UFS],
                     lambda uf, ano: DIR_PARCIAIS / "cnes_in" / f"IN_{uf}_{ano}.parquet",
                     incentivos_uf_ano)
ARQ_INEXISTENTES_IN = DIR_SAIDA / "cnes_in_arquivos_inexistentes.csv"
if INEXISTENTES_IN:
    anteriores = pd.read_csv(ARQ_INEXISTENTES_IN) if ARQ_INEXISTENTES_IN.exists() else pd.DataFrame()
    (pd.concat([anteriores, pd.DataFrame(INEXISTENTES_IN)]).drop_duplicates()
       .sort_values(["ano", "uf"]).to_csv(ARQ_INEXISTENTES_IN, index=False))
if ARQ_INEXISTENTES_IN.exists():
    inexistentes = pd.read_csv(ARQ_INEXISTENTES_IN)
    print(f"UF-anos sem arquivo CNES-IN no FTP (tratados como zero incentivos): {len(inexistentes)}")
    display(inexistentes.groupby("ano")["uf"].agg(lambda u: ", ".join(sorted(u))).to_frame("UFs").T)
if frequencias_in:
    print(f"Códigos de {COL_INCENTIVO} mais frequentes ({UFS[0]}, dez/{ANOS[-1]}):")
    display(frequencias_in[0].rename("estabelecimentos").to_frame().T)
if len(incentivos):
    display(incentivos.groupby("ano")["n_estab_incentivo"].agg(municipios="count", estab="sum").T)
else:
    print("Nenhum estabelecimento com os códigos", sorted(CODIGOS_INCENTIVO), "— conferir o formato acima.")

CNES-IN: 459 de 459 partes já gravadas (lidas do disco); 0 a baixar
CNES-IN: 25/459 partes | 0.0 min | falhas: 0
CNES-IN: 50/459 partes | 0.0 min | falhas: 0
CNES-IN: 75/459 partes | 0.0 min | falhas: 0
CNES-IN: 100/459 partes | 0.0 min | falhas: 0
CNES-IN: 125/459 partes | 0.0 min | falhas: 0
CNES-IN: 150/459 partes | 0.0 min | falhas: 0
CNES-IN: 175/459 partes | 0.0 min | falhas: 0
CNES-IN: 200/459 partes | 0.0 min | falhas: 0
CNES-IN: 225/459 partes | 0.0 min | falhas: 0
CNES-IN: 250/459 partes | 0.0 min | falhas: 0
CNES-IN: 275/459 partes | 0.0 min | falhas: 0
CNES-IN: 300/459 partes | 0.0 min | falhas: 0
CNES-IN: 325/459 partes | 0.0 min | falhas: 0
CNES-IN: 350/459 partes | 0.0 min | falhas: 0
CNES-IN: 375/459 partes | 0.0 min | falhas: 0
CNES-IN: 400/459 partes | 0.0 min | falhas: 0
CNES-IN: 425/459 partes | 0.0 min | falhas: 0
CNES-IN: 450/459 partes | 0.0 min | falhas: 0
CNES-IN: 459/459 partes | 0.0 min | falhas: 0
UF-anos sem arquivo CNES-IN no FTP (tratados como zero incent

ano,2008,2009,2010,2011
UFs,"DF, RR","DF, RR",DF,DF


ano,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
municipios,23,23,91,91,91,90,90,91,91,91,91
estab,26,26,95,95,95,94,93,94,94,94,94


**Resultado:** [PREENCHER: se 81.12/81.13 aparecem entre os códigos (e em que formato) e quantos municípios têm o incentivo vigente por ano. Os códigos mais frequentes só são exibidos quando a parte é baixada, e não lida do checkpoint.]

#### 3.3.1 Quais códigos de incentivo os polos tipo 74 têm?
**Por quê:** no teste (AC, dez/2015), os códigos 81.12/81.13 não apareceram. Em vez de supor o código, esta célula cruza, para **todas as UFs** em dez/2016 (ano com cerca de 1.900 polos), os códigos de `SGRUPHAB` vigentes com o tipo de estabelecimento do CNES-ST, e mostra os códigos mais frequentes entre os polos tipo 74. Os autores confirmam o código na documentação do CNES e ajustam `CODIGOS_INCENTIVO` acima. São 54 arquivos pequenos (IN e ST de cada UF), gravados em checkpoint.

In [47]:
ANO_DIAGNOSTICO_IN = 2016


def diagnostico_in_uf(uf):
    inc = dl.baixar_datasus("cnes", uf, ANO_DIAGNOSTICO_IN, MES_CNES, grupo="IN", colunas=COLS_IN)
    st = dl.baixar_datasus("cnes", uf, ANO_DIAGNOSTICO_IN, MES_CNES, grupo="ST", colunas=["CNES", "TP_UNID"])
    return ag.codigos_incentivo_por_tipo(inc, st, ANO_DIAGNOSTICO_IN * 100 + MES_CNES).assign(uf=uf)


diag_in = extrair("Diagnóstico IN", [(uf,) for uf in UFS_TODAS],
                  lambda uf: DIR_PROC / "diagnosticos" / f"IN_ST_{uf}_{ANO_DIAGNOSTICO_IN}.parquet",
                  diagnostico_in_uf)
codigos = (diag_in.groupby("codigo")[["n_estab", "n_estab_tipo74"]].sum()
           .assign(pct_tipo74=lambda d: (100 * d["n_estab_tipo74"] / d["n_estab"]).round(1))
           .sort_values("n_estab_tipo74", ascending=False))
codigos.to_csv(DIR_PROC / "diagnosticos" / f"codigos_incentivo_tipo74_{ANO_DIAGNOSTICO_IN}.csv")
print(f"Códigos vigentes em dez/{ANO_DIAGNOSTICO_IN} mais frequentes entre os polos tipo 74 (Brasil):")
codigos.head(10)

Diagnóstico IN: 27 de 27 partes já gravadas (lidas do disco); 0 a baixar
Diagnóstico IN: 25/27 partes | 0.0 min | falhas: 0
Diagnóstico IN: 27/27 partes | 0.0 min | falhas: 0
Códigos vigentes em dez/2016 mais frequentes entre os polos tipo 74 (Brasil):


,n_estab,n_estab_tipo74,pct_tipo74
codigo,,,
8112,95,94,98.9
8107,336,13,3.9
8101,742,0,0.0
8102,18,0,0.0
8103,16,0,0.0
8105,93,0,0.0
8104,270,0,0.0
8111,2,0,0.0
8201,49,0,0.0


**Execução de 22/11/2026:** o código `8112` está vigente em 95 estabelecimentos, 94 deles polos tipo 74 (98,9%); `8107` em 336, 13 deles tipo 74; `8113` não aparece. Só cerca de 5% dos polos tipo 74 de dez/2016 têm `8112`.

**Resultado:** [PREENCHER: códigos com maior número e percentual de polos tipo 74; conferir na documentação do CNES (tabela de habilitações/incentivos) qual corresponde ao custeio do Programa Academia da Saúde e ajustar `CODIGOS_INCENTIVO` na célula anterior. Depois de mudar os códigos, apagar a pasta `parciais/cnes_in` para recalcular.]

### 3.4 CNES-EP e CNES-LT, dezembro de 2010: equipes e leitos (covariáveis)
**Por quê:** confundidores medidos antes do tratamento (PLANO.md, seção 4.2): equipes de atenção básica e leitos SUS por município. As equipes são contadas por tipo (`TIPO_EQP`), excluindo só as com data de desativação válida até dez/2010 (valores vazios ou que não são datas contam como ativas); a escolha de quais tipos correspondem a Saúde da Família e NASF é feita no caderno 02, a partir dos rótulos.

In [48]:
def equipes_uf(uf):
    ep = dl.baixar_datasus("cnes", uf, ANO_COVARIAVEIS, MES_CNES, grupo="EP",
                           colunas=["CODUFMUN", "TIPO_EQP", "DT_DESAT"])
    if MODO_TESTE:
        print(f"EP {uf} dez/{ANO_COVARIAVEIS}: {len(ep)} equipes lidas")
    return ag.contar_equipes(ep, data_referencia=f"{ANO_COVARIAVEIS}-12-31")


def leitos_uf(uf):
    lt = dl.baixar_datasus("cnes", uf, ANO_COVARIAVEIS, MES_CNES, grupo="LT",
                           colunas=["CODUFMUN", "QT_SUS", "QT_EXIST"])
    return ag.somar_leitos(lt)


equipes = extrair("CNES-EP", [(uf,) for uf in UFS],
                  lambda uf: DIR_PARCIAIS / "cnes_ep" / f"EP_{uf}_{ANO_COVARIAVEIS}_v2.parquet", equipes_uf)
leitos = extrair("CNES-LT", [(uf,) for uf in UFS],
                 lambda uf: DIR_PARCIAIS / "cnes_lt" / f"LT_{uf}_{ANO_COVARIAVEIS}.parquet", leitos_uf)
if not len(equipes):
    print("ATENÇÃO: nenhuma equipe ativa — conferir TIPO_EQP e DT_DESAT no dicionário cnes_ep_colunas.csv.")
else:
    display(equipes.groupby("tipo_eqp")["n_equipes"].sum().sort_values(ascending=False).head(15).to_frame().T)
if len(leitos):
    assert leitos["cod6"].is_unique
    print("Leitos SUS (soma):", int(leitos["leitos_sus"].sum()), "| municípios com leito:", len(leitos))

CNES-EP: 27 de 27 partes já gravadas (lidas do disco); 0 a baixar
CNES-EP: 25/27 partes | 0.0 min | falhas: 0
CNES-EP: 27/27 partes | 0.0 min | falhas: 0
CNES-LT: 27 de 27 partes já gravadas (lidas do disco); 0 a baixar
CNES-LT: 25/27 partes | 0.0 min | falhas: 0
CNES-LT: 27/27 partes | 0.0 min | falhas: 0


tipo_eqp,02,01,04,03,06,05,10,07,08,11,09
n_equipes,19876,10664,3736,1913,1261,273,255,151,43,24,12


Leitos SUS (soma): 359968 | municípios com leito: 3691


**Resultado:** [PREENCHER: tipos de equipe mais frequentes e total de leitos SUS em dez/2010]

### 3.5 IBGE/SIDRA: população, PIB 2010 e população idosa 2010
**Por quê:** população é o denominador das taxas; PIB per capita e % de 60 anos ou mais são confundidores (PLANO.md, seção 4.2).

- **População:** tabela 6579 (estimativas), variável 9324, consultada ano a ano. A lista de anos publicados vem da própria API; anos sem estimativa (anos de Censo, por exemplo) são interpolados na seção 6.
- **PIB municipal 2010:** tabela 5938, variável 37. A célula confere pelo nome que a variável é o PIB a preços correntes.
- **População idosa 2010:** tabela 1378 do Censo 2010, variável 93. As categorias de idade são lidas dos metadados da tabela, e a soma das idades simples é conferida com o total no nível Brasil antes da consulta por município.

In [49]:
def populacao_ano(ano):
    tab = dl.tabela_sidra("6579", "9324", str(ano))
    return pd.DataFrame({"cod_ibge7": tab["Município (Código)"], "ano": ano,
                         "populacao": pd.to_numeric(tab["Valor"], errors="coerce")})


anos_publicados = {int(p) for p in dl.periodos_sidra("6579")}
anos_sem_estimativa = sorted(set(ANOS) - anos_publicados)
print("Anos sem estimativa na tabela 6579 (serão interpolados na seção 6):", anos_sem_estimativa)
pop_sidra = extrair("SIDRA-6579", [(ano,) for ano in ANOS if ano in anos_publicados],
                    lambda ano: DIR_PARCIAIS / "sidra_pop" / f"POP_{ano}.parquet", populacao_ano)


def pib_2010():
    tab = dl.tabela_sidra("5938", "37", str(ANO_COVARIAVEIS))
    assert tab["Variável"].str.contains("Produto Interno Bruto a preços correntes").all(), \
        tab["Variável"].unique()
    return pd.DataFrame({"cod_ibge7": tab["Município (Código)"],
                         "pib_mil_reais": pd.to_numeric(tab["Valor"], errors="coerce")})


pib = com_checkpoint(DIR_SAIDA / "sidra_pib_2010.parquet", pib_2010)
codigos_uf = sorted(municipios["cod_ibge7"].str[:2].unique())
idosos = com_checkpoint(DIR_SAIDA / "sidra_idosos_2010.parquet",
                        lambda: dl.populacao_idosos_sidra("1378", "93", "2010", codigos_uf))
print("PIB 2010:", len(pib), "municípios | idosos 2010:", len(idosos), "municípios")

Anos sem estimativa na tabela 6579 (serão interpolados na seção 6): [2010, 2022, 2023]
SIDRA-6579: 14 de 14 partes já gravadas (lidas do disco); 0 a baixar
SIDRA-6579: 14/14 partes | 0.0 min | falhas: 0
PIB 2010: 5570 municípios | idosos 2010: 5565 municípios


**Resultado:** [PREENCHER: anos sem estimativa populacional; número de municípios com PIB e com população idosa em 2010]

### 3.6 SIH/SUS-RD: internações por município de residência × ano × grupo de CID
**Por quê:** constrói os desfechos (hipertensão, AVC, diabetes), o placebo (apendicite) e o total ("outras"), por município de **residência** (`MUNIC_RES`). Lê só `MUNIC_RES`, `DIAG_PRINC`, `ANO_CMPT` e `IDENT`; mantém o tipo de AIH (`IDENT`) para a decisão da seção 6.

Cada parte é uma UF **de internação** em um ano (12 arquivos mensais); como a contagem é por município de residência, as partes são somadas no fim (residentes internados em outras UFs aparecem em mais de uma parte). Se algum mês falhar, a parte inteira não é gravada e será refeita na próxima execução. Esta é a etapa mais demorada: no modo completo são cerca de 5.500 arquivos.

In [50]:
COLS_RD = ["MUNIC_RES", "DIAG_PRINC", "ANO_CMPT", "IDENT"]


def sih_uf_ano(uf, ano):
    partes = []
    for mes in range(1, 13):
        rd = dl.baixar_datasus("sih", uf, ano, mes, grupo="RD", colunas=COLS_RD)
        assert {"MUNIC_RES", "DIAG_PRINC", "ANO_CMPT"} <= set(rd.columns), f"RD {uf} {ano}/{mes}"
        partes.append(ag.agregar_sih(rd))
    return (pd.concat(partes, ignore_index=True)
              .groupby(["cod6", "ano", "grupo", "ident"], as_index=False)["internacoes"].sum())


sih = extrair("SIH-RD", [(uf, ano) for ano in ANOS for uf in UFS],
              lambda uf, ano: DIR_PARCIAIS / "sih_rd" / f"RD_{uf}_{ano}.parquet", sih_uf_ano)
if len(sih):
    # Cada parte é uma UF de internação; residentes de um município internados em
    # outras UFs aparecem em mais de uma parte. Soma-se tudo por município de residência.
    n_linhas_partes = len(sih)
    sih = sih.groupby(["cod6", "ano", "grupo", "ident"], as_index=False)["internacoes"].sum()
    assert not sih.duplicated(["cod6", "ano", "grupo", "ident"]).any()
    print(f"Linhas das partes: {n_linhas_partes:,} -> após somar por residência: {len(sih):,}")
    display(sih.pivot_table(index="ano", columns="grupo", values="internacoes", aggfunc="sum"))

SIH-RD: 459 de 459 partes já gravadas (lidas do disco); 0 a baixar
SIH-RD: 25/459 partes | 0.0 min | falhas: 0
SIH-RD: 50/459 partes | 0.0 min | falhas: 0
SIH-RD: 75/459 partes | 0.0 min | falhas: 0
SIH-RD: 100/459 partes | 0.0 min | falhas: 0
SIH-RD: 125/459 partes | 0.0 min | falhas: 0
SIH-RD: 150/459 partes | 0.0 min | falhas: 0
SIH-RD: 175/459 partes | 0.0 min | falhas: 0
SIH-RD: 200/459 partes | 0.0 min | falhas: 0
SIH-RD: 225/459 partes | 0.0 min | falhas: 0
SIH-RD: 250/459 partes | 0.0 min | falhas: 0
SIH-RD: 275/459 partes | 0.0 min | falhas: 0
SIH-RD: 300/459 partes | 0.0 min | falhas: 0
SIH-RD: 325/459 partes | 0.0 min | falhas: 0
SIH-RD: 350/459 partes | 0.0 min | falhas: 0
SIH-RD: 375/459 partes | 0.0 min | falhas: 0
SIH-RD: 400/459 partes | 0.0 min | falhas: 0
SIH-RD: 425/459 partes | 0.0 min | falhas: 0
SIH-RD: 450/459 partes | 0.0 min | falhas: 0
SIH-RD: 459/459 partes | 0.0 min | falhas: 0
Linhas das partes: 759,362 -> após somar por residência: 500,442


grupo,apendicite,avc,diabetes,hipertensao,outras
ano,,,,,
2008,81355,163600,131734,142637,10587829
2009,93713,182078,141503,141453,10957352
2010,97206,190087,148452,134059,11155030
2011,96902,196660,148511,127389,11074006
2012,102667,194694,142677,117592,10882259
2013,107799,200005,140873,109165,10962995
2014,111910,206697,139819,105226,11049063
2015,115843,212097,138435,95859,11076619
2016,115927,216087,128585,83688,10983425


**Resultado:** [PREENCHER: totais por ano e grupo, antes do filtro por tipo de AIH]

## 4. Descompressão e seleção

O PySUS baixa cada arquivo do FTP do DATASUS (`.dbc`, compactado), descompacta e converte para parquet no cache `data/raw/pysus/`. De cada arquivo são lidas **apenas** as colunas listadas em `COLS_ST`, `COL_INCENTIVO`, nas chamadas do CNES-EP/LT e em `COLS_RD`; o arquivo bruto é apagado logo após a leitura. Ficam guardadas só as bases agregadas listadas abaixo.

In [51]:
mantidos = pd.DataFrame(
    [(p.parent.name, p.name) for p in sorted(DIR_SAIDA.rglob("*.parquet"))],
    columns=["pasta", "arquivo"])
print("Arquivos agregados mantidos:", len(mantidos))
display(mantidos.groupby("pasta").size().rename("arquivos").to_frame().T)
restantes = [p for p in (DIR_RAW / "pysus").rglob("*") if p.is_file() and not p.name.startswith(".")]
print("Arquivos restantes no cache do PySUS:", len(restantes))
if restantes:
    display(pd.DataFrame({"extensao": [p.suffix or "(sem)" for p in restantes],
                          "mb": [p.stat().st_size / 1e6 for p in restantes]})
            .groupby("extensao")["mb"].agg(arquivos="count", mb="sum").round(2).T)

Arquivos agregados mantidos: 1493


pasta,cnes_ep,cnes_in,cnes_lt,cnes_st,diagnosticos,processed,sidra_pop,sih_rd,teste
arquivos,29,461,28,461,27,5,16,461,5


Arquivos restantes no cache do PySUS: 5


extensao,.db,.duckdb
arquivos,1.00,4.00
mb,1.06,222.35


**Resultado:** [PREENCHER: número de arquivos agregados por pasta. No cache devem restar só arquivos de controle do PySUS (ex.: banco `.db`), não `.dbc` nem `.parquet` de dados brutos]

## 5. Dicionário de variáveis e rótulos de valores

**O que:** para cada base bruta do DATASUS, exporta `data/processed/dicionarios/<base>_colunas.csv` com o nome e o tipo de **todas** as colunas do arquivo original (lidos do esquema, sem carregar valores), marcando as usadas e descrevendo-as. Exporta também os rótulos de valores usados nas regras de construção.
**Por quê:** etapa 5 das orientações — documentar o dado bruto antes da transformação. As descrições abaixo cobrem só as colunas usadas e devem ser conferidas na documentação do DATASUS.

In [52]:
DESCRICOES = {
    "CNES": "Código do estabelecimento no CNES",
    "CODUFMUN": "Município do estabelecimento (código IBGE de 6 dígitos)",
    "TP_UNID": "Tipo de estabelecimento (74 = Polo Academia da Saúde)",
    "TIPO_EQP": "Tipo de equipe de saúde",
    "DT_DESAT": "Data de desativação da equipe",
    "QT_SUS": "Quantidade de leitos SUS",
    "QT_EXIST": "Quantidade de leitos existentes",
    "MUNIC_RES": "Município de residência do paciente (código IBGE de 6 dígitos)",
    "DIAG_PRINC": "Diagnóstico principal (CID-10)",
    "ANO_CMPT": "Ano de competência (processamento) da AIH",
    "IDENT": "Identificação do tipo de AIH",
}
DESCRICOES.update({
    "SGRUPHAB": "Código do subgrupo de habilitação/incentivo (conferir na documentação do CNES)",
    "CMPT_INI": "Competência inicial da habilitação/incentivo (AAAAMM)",
    "CMPT_FIM": "Competência final da habilitação/incentivo (AAAAMM)",
})

AMOSTRAS = {  # base: (sistema, uf, ano, mes, grupo) — um arquivo representativo de cada base
    "cnes_st": ("cnes", UFS[0], ANO_COVARIAVEIS, MES_CNES, "ST"),
    "cnes_in": ("cnes", UFS[0], ANO_COVARIAVEIS, MES_CNES, "IN"),
    "cnes_ep": ("cnes", UFS[0], ANO_COVARIAVEIS, MES_CNES, "EP"),
    "cnes_lt": ("cnes", UFS[0], ANO_COVARIAVEIS, MES_CNES, "LT"),
    "sih_rd": ("sih", UFS[0], ANOS[-1], 1, "RD"),
}
for base, args in AMOSTRAS.items():
    destino = DIR_DIC / f"{base}_colunas.csv"
    if not destino.exists():
        dic = dl.esquema_datasus(*args)
        dic["usada"] = dic["coluna"].isin(DESCRICOES)
        dic["descricao"] = dic["coluna"].map(DESCRICOES).fillna("")
        dic["arquivo_de_referencia"] = "{}-{} {} {}/{}".format(args[0].upper(), args[4], *args[1:4])
        dic.to_csv(destino, index=False)
    print(base, "->", destino.name, "|", len(pd.read_csv(destino)), "colunas")

rotulos = pd.DataFrame(
    [("TP_UNID", "74", "Polo Academia da Saúde"),
     ("IDENT", "1", "AIH normal"),
     ("IDENT", "5", "AIH de longa permanência")]
    + [("grupo_cid", g, f"CID-10 {ini}–{fim}") for g, (ini, fim) in ag.GRUPOS_CID.items()]
    + [("grupo_cid", ag.GRUPO_OUTRAS, "Demais diagnósticos")]
    + [("incentivo", c, "Incentivo de custeio do Programa Academia da Saúde") for c in sorted(CODIGOS_INCENTIVO)],
    columns=["variavel", "valor", "rotulo"])
rotulos.to_csv(DIR_DIC / "rotulos_valores.csv", index=False)
rotulos

cnes_st -> cnes_st_colunas.csv | 200 colunas
cnes_in -> cnes_in_colunas.csv | 30 colunas
cnes_ep -> cnes_ep_colunas.csv | 100 colunas
cnes_lt -> cnes_lt_colunas.csv | 27 colunas
sih_rd -> sih_rd_colunas.csv | 113 colunas


,variavel,valor,rotulo
0,TP_UNID,74,Polo Academia da Saúde
1,IDENT,1,AIH normal
2,IDENT,5,AIH de longa permanência
3,grupo_cid,hipertensao,CID-10 I10–I15
4,grupo_cid,avc,CID-10 I60–I69
5,grupo_cid,diabetes,CID-10 E10–E14
6,grupo_cid,apendicite,CID-10 K35–K38
7,grupo_cid,outras,Demais diagnósticos
8,incentivo,81.12,Incentivo de custeio do Programa Academia da S...
9,incentivo,81.13,Incentivo de custeio do Programa Academia da S...


**Resultado:** dicionários e rótulos gravados em `data/processed/dicionarios/`. [PREENCHER: conferir os rótulos de `IDENT` e dos incentivos na documentação do DATASUS]

## 6. Agregação e validação

### 6.1 Pendências da extração
Se alguma parte falhou (por exemplo, queda de conexão), a construção do painel é interrompida. Basta executar o caderno de novo: as partes já gravadas são lidas do disco e só as que falharam são baixadas.

In [53]:
falhas = pd.DataFrame(FALHAS, columns=["base", "parte", "erro"])
falhas.to_csv(DIR_SAIDA / "falhas_extracao.csv", index=False)
assert falhas.empty, f"{len(falhas)} partes falharam (ver abaixo). Execute o caderno de novo.\n{falhas.head(10)}"
print("Nenhuma falha: todas as partes foram extraídas.")

Nenhuma falha: todas as partes foram extraídas.


Sem falhas, todas as partes de todas as bases estão disponíveis.

### 6.2 Tipo de AIH
**Decisão:** contar só as AIHs normais (`IDENT == "1"`), excluindo as de longa permanência (`IDENT == "5"`), que são continuações de internações já contadas. Essa é a definição usada pelo TabNet para "internações" — a conferência da seção 6.5 confirma ou não. A célula mostra quantas AIHs saem.

In [54]:
por_ident = sih.groupby("ident")["internacoes"].sum()
display(por_ident.rename("AIHs").to_frame().T)
internacoes = (sih.loc[sih["ident"] == "1"]
                  .groupby(["cod6", "ano", "grupo"], as_index=False)["internacoes"].sum())
print(f"AIHs excluídas (IDENT != 1): {int(por_ident.drop('1', errors='ignore').sum()):,} "
      f"de {int(por_ident.sum()):,}")

ident,1,5
AIHs,197914729,4214523


AIHs excluídas (IDENT != 1): 4,214,523 de 202,129,252


**Resultado:** [PREENCHER: número e percentual de AIHs excluídas]

### 6.3 Distrito Federal e municípios de residência fora da lista do IBGE
**Decisão (Distrito Federal):** o DF é um único município no IBGE (Brasília, `530010`), mas o DATASUS registra parte das ocorrências com códigos de regiões administrativas (`530040`, `530180` etc.). Todos os códigos `53xxxx` são unificados em `530010`, nas internações e nas bases do CNES (polos, incentivos, leitos e equipes). A célula mostra quantas internações foram reatribuídas.

**Por quê (demais códigos):** `MUNIC_RES` pode trazer códigos que não correspondem a municípios da lista (ignorados, exterior ou códigos antigos). Essas internações não entram no painel; a célula conta quantas são.

In [55]:
codigos_df = internacoes["cod6"].str.startswith("53") & (internacoes["cod6"] != ag.COD6_BRASILIA)
print(f"Internações com código de região administrativa do DF reatribuídas a Brasília: "
      f"{int(internacoes.loc[codigos_df, 'internacoes'].sum()):,}")
internacoes = ag.somar_por_municipio(internacoes, ["cod6", "ano", "grupo"], ["internacoes"])
polos = ag.somar_por_municipio(polos, ["cod6", "ano"], ["n_polos"]) if len(polos) else polos
incentivos = (ag.somar_por_municipio(incentivos, ["cod6", "ano"], ["n_estab_incentivo"])
              if len(incentivos) else incentivos)
leitos = ag.somar_por_municipio(leitos, ["cod6"], ["leitos_sus", "leitos_existentes"]) if len(leitos) else leitos
equipes = ag.somar_por_municipio(equipes, ["cod6", "tipo_eqp"], ["n_equipes"]) if len(equipes) else equipes

fora = ~internacoes["cod6"].isin(municipios["cod6"])
if MODO_TESTE:  # no teste, residentes de outras UFs internados no AC também ficam fora
    print("Modo teste: inclui residentes de outras UFs.")
print(f"Internações com município fora da lista: {int(internacoes.loc[fora, 'internacoes'].sum()):,} "
      f"({internacoes.loc[fora, 'internacoes'].sum() / internacoes['internacoes'].sum():.2%})")
display(internacoes.loc[fora].groupby("cod6")["internacoes"].sum().nlargest(5).to_frame().T)
internacoes = internacoes.loc[~fora]

Internações com código de região administrativa do DF reatribuídas a Brasília: 1,270,397
Internações com município fora da lista: 0 (0.00%)


cod6
internacoes


**Resultado:** [PREENCHER: internações do DF reatribuídas; número e percentual de internações descartadas e principais códigos. Na execução de 24/11/2026, antes da unificação do DF, 1.270.397 internações (0,64%) ficavam fora, a maior parte com códigos do DF]

### 6.4 População e painel município × ano
**Decisão:** anos sem estimativa populacional na tabela 6579 são interpolados linearmente entre os anos vizinhos, por município (`populacao_imputada = True`). Não há extrapolação: municípios instalados durante o período ficam sem população nos anos anteriores à instalação, e o caderno 02 decide se saem da amostra.

O painel tem uma linha por município da lista e por ano; contagens ausentes (nenhum polo, nenhuma internação) valem zero.

In [56]:
pop = pop_sidra.merge(municipios[["cod_ibge7", "cod6"]], on="cod_ibge7", how="inner")
pop = ag.interpolar_populacao(pop[["cod6", "ano", "populacao"]], ANOS)
painel = ag.montar_painel(municipios[["cod6", "cod_ibge7", "municipio", "uf", "regiao"]],
                          ANOS, polos, internacoes, pop)

n_mun, n_anos = len(municipios), len(ANOS)
assert len(painel) == n_mun * n_anos, (len(painel), n_mun * n_anos)
assert not painel.duplicated(["cod6", "ano"]).any()
colunas_contagem = ["n_polos"] + [c for c in painel.columns if c.startswith("int_")]
assert painel[colunas_contagem].notna().all().all()
assert (painel[colunas_contagem] >= 0).all().all()

# População ausente só pode ocorrer nos primeiros anos (município ainda não instalado).
sem_pop = painel.loc[painel["populacao"].isna()]
for cod, anos_faltantes in sem_pop.groupby("cod6")["ano"]:
    esperado = ANOS[: len(anos_faltantes)]
    assert sorted(anos_faltantes) == esperado, f"{cod}: população ausente fora do início: {sorted(anos_faltantes)}"
print(f"Painel: {len(painel):,} linhas ({n_mun} municípios × {n_anos} anos)")
print("Municípios sem população em algum ano (instalados durante o período):", sem_pop["cod6"].nunique())
print("Anos com população interpolada:", sorted(painel.loc[painel["populacao_imputada"] == True, "ano"].unique()))
painel.head(3)

Painel: 94,707 linhas (5571 municípios × 17 anos)
Municípios sem população em algum ano (instalados durante o período): 6
Anos com população interpolada: [np.int64(2010), np.int64(2022), np.int64(2023)]


,cod6,ano,cod_ibge7,municipio,uf,regiao,n_polos,int_hipertensao,int_avc,int_diabetes,int_apendicite,int_outras,populacao,populacao_imputada
0,110001,2008,1100015,Alta Floresta D'Oeste,RO,Norte,0,132,9,54,5,2309,24577.0,False
1,110001,2009,1100015,Alta Floresta D'Oeste,RO,Norte,0,94,10,50,7,2419,24354.0,False
2,110001,2010,1100015,Alta Floresta D'Oeste,RO,Norte,0,90,8,58,3,2403,24291.0,True


**Resultado:** [PREENCHER: número de linhas, municípios sem população em algum ano e anos interpolados]

### 6.5 Conferência com o TabNet e coortes
**O que:** totais por ano e grupo, e os números de **um município-ano** para comparar com o TabNet do DATASUS (Morbidade Hospitalar do SUS por local de residência, ano de processamento, itens da Lista Morb CID-10: hipertensão = 169 + 170; AVC = 177 a 180; diabetes = 124; apendicite = 210).
**Por quê:** validar a agregação antes de usar a base (PLANO.md, Fase 3, revisão crítica).

**Atenção ao recorte.** No `MODO_TESTE`, o caderno só lê os arquivos do AC; compare com o TabNet **filtrando os estabelecimentos do Acre**. No modo completo, compare sem filtro de estabelecimento. A conferência de 22/11/2026 (AC, 2014–2015) bateu exatamente nos quatro grupos e no total: ver `docs/validacao/tabnet_2026-11-22.md` e o script `docs/validacao/consulta_tabnet.py`.

In [57]:
totais = painel.groupby("ano")[[c for c in painel.columns if c.startswith("int_")]].sum()
display(totais)
MUN_CONFERENCIA = "120040" if "120040" in set(painel["cod6"]) else painel["cod6"].iloc[0]  # Rio Branco/AC
display(painel.loc[(painel["cod6"] == MUN_CONFERENCIA) & (painel["ano"] == ANOS[-1]),
                   ["cod6", "municipio", "ano"] + [c for c in painel.columns if c.startswith("int_")]])

coorte = ag.primeiro_ano_tratamento(painel)
print("Municípios por coorte g (primeiro ano com polo tipo 74):")
display(coorte.value_counts().sort_index().rename("municípios").to_frame().T)
print(f"Com polo antes de {ANO_PRIMEIRA_COORTE} (saem da amostra principal no caderno 02):",
      int((coorte < ANO_PRIMEIRA_COORTE).sum()))

,int_hipertensao,int_avc,int_diabetes,int_apendicite,int_outras
ano,,,,,
2008,141252,156867,131734,81355,10232395
2009,139479,175022,141503,93713,10583632
2010,131809,183621,148452,97206,10796877
2011,124869,190441,148511,96902,10720848
2012,114918,188667,142677,102667,10543660
2013,106180,194213,140873,107799,10648095
2014,102222,201497,139819,111910,10764839
2015,93008,207147,138435,115843,10823283
2016,81300,211534,128582,115927,10759725


,cod6,municipio,ano,int_hipertensao,int_avc,int_diabetes,int_apendicite,int_outras
1138,120040,Rio Branco,2024,66,367,302,292,23546


Municípios por coorte g (primeiro ano com polo tipo 74):


g,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
municípios,300,24,206,397,376,290,247,228,168,141,148,128,73,63


Com polo antes de 2013 (saem da amostra principal no caderno 02): 324


**Resultado:** [PREENCHER: valores do município-ano conferidos no TabNet, com a data da consulta. No teste de 22/11/2026, Rio Branco 2015 (hospitais do AC): hipertensão 99, AVC 281, diabetes 149, apendicite 377 e total 22.706, iguais no caderno e no TabNet. No modo completo, repetir sem filtro de estabelecimento.]

### 6.6 Covariáveis de 2010
Junta, por município: população do Censo 2010, PIB per capita, % de 60 anos ou mais, leitos SUS por mil habitantes e equipes por tipo. A taxa de internação pré-tratamento é calculada no caderno 02, a partir do painel.

In [58]:
covariaveis = (municipios[["cod6", "cod_ibge7", "uf", "regiao"]]
               .merge(pib, on="cod_ibge7", how="left")
               .merge(idosos, on="cod_ibge7", how="left")
               .merge(leitos, on="cod6", how="left"))
numericas = ["pib_mil_reais", "pop_total_censo", "pop_idosos", "leitos_sus", "leitos_existentes"]
covariaveis[numericas] = covariaveis[numericas].apply(pd.to_numeric, errors="coerce").astype("float64")
covariaveis[["leitos_sus", "leitos_existentes"]] = covariaveis[["leitos_sus", "leitos_existentes"]].fillna(0)
covariaveis["pib_per_capita"] = covariaveis["pib_mil_reais"] * 1000 / covariaveis["pop_total_censo"]
covariaveis["pct_60_mais"] = 100 * covariaveis["pop_idosos"] / covariaveis["pop_total_censo"]
covariaveis["leitos_sus_por_mil"] = 1000 * covariaveis["leitos_sus"] / covariaveis["pop_total_censo"]

equipes_largo = (equipes.pivot_table(index="cod6", columns="tipo_eqp", values="n_equipes",
                                     aggfunc="sum", fill_value=0)
                 .add_prefix("equipes_tipo_").reset_index()) if len(equipes) else pd.DataFrame({"cod6": []})
covariaveis = covariaveis.merge(equipes_largo, on="cod6", how="left")
colunas_eq = [c for c in covariaveis.columns if c.startswith("equipes_tipo_")]
covariaveis[colunas_eq] = covariaveis[colunas_eq].fillna(0).astype("int64")

assert covariaveis["cod6"].is_unique and len(covariaveis) == n_mun
faltantes = covariaveis.isna().sum()
print("Faltantes por coluna (municípios instalados depois de 2010 não têm Censo 2010):")
display(faltantes[faltantes > 0].to_frame("faltantes").T)
covariaveis.describe().T.head(12)

Faltantes por coluna (municípios instalados depois de 2010 não têm Censo 2010):


,pib_mil_reais,pop_total_censo,pop_idosos,pib_per_capita,pct_60_mais,leitos_sus_por_mil
faltantes,6,6,6,6,6,6


,count,mean,std,min,25%,50%,75%,max
pib_mil_reais,5565.0,698265.403774,7.396506e+06,7218.000000,41213.000000,90690.000000,238758.000000,4.504920e+08
pop_total_censo,5565.0,34277.771608,2.031126e+05,805.000000,5235.000000,10934.000000,23424.000000,1.125350e+07
pop_idosos,5565.0,3700.017430,2.445243e+04,86.000000,646.000000,1279.000000,2567.000000,1.338138e+06
leitos_sus,5571.0,64.614611,4.057959e+02,0.000000,0.000000,19.000000,42.000000,1.824200e+04
leitos_existentes,5571.0,91.152755,7.142081e+02,0.000000,0.000000,20.000000,49.000000,3.526300e+04
pib_per_capita,5565.0,12603.481102,1.471985e+04,2261.646586,5130.936175,9301.059743,15171.390013,3.118820e+05
pct_60_mais,5565.0,12.087828,3.275984e+00,2.595458,9.911504,12.073535,14.162605,2.938482e+01
leitos_sus_por_mil,5565.0,1.572340,1.856513e+00,0.000000,0.000000,1.256084,2.316781,2.454400e+01
equipes_tipo_01,5571.0,1.914199,1.242014e+01,0.000000,0.000000,0.000000,2.000000,7.460000e+02
equipes_tipo_02,5571.0,3.567762,7.744242e+00,0.000000,1.000000,2.000000,4.000000,1.840000e+02


**Resultado:** [PREENCHER: faltantes por covariável e faixas plausíveis (PIB per capita, % de idosos, leitos por mil)]

## 7. Gravação em `data/processed/`

Grava as bases usadas pelo caderno 02 e o registro de acessos (fontes, origem efetiva e data/hora de cada download). No Colab, a pasta fica no Google Drive (`APPD_26_dados/processed`).

In [59]:
painel.to_parquet(DIR_SAIDA / "painel_bruto.parquet", index=False)
covariaveis.to_csv(DIR_SAIDA / "covariaveis_2010.csv", index=False)
equipes.to_csv(DIR_SAIDA / "equipes_2010_por_tipo.csv", index=False)
incentivos.to_parquet(DIR_SAIDA / "incentivos_custeio.parquet", index=False)
dl.registro_acessos().to_csv(DIR_SAIDA / "registro_acessos.csv", index=False)

for nome in ["painel_bruto.parquet", "covariaveis_2010.csv", "equipes_2010_por_tipo.csv",
             "incentivos_custeio.parquet", "registro_acessos.csv"]:
    assert (DIR_SAIDA / nome).exists(), nome
print(f"Bases gravadas em {DIR_SAIDA.relative_to(RAIZ)}/")
print(f"Tempo desta execução: {(time.time() - INICIO) / 60:.1f} min")

Bases gravadas em data/processed/
Tempo desta execução: 0.2 min


O tempo desta execução deve ser copiado para o `README.md` quando for a extração completa feita do zero (sem checkpoints).